In [0]:
# Create sample dataframe and write as Delta
data = [(1, "alice", 100), (2, "bob", 200)]
cols = ["id","name","amt"]
df = spark.createDataFrame(data, cols)

In [0]:
display(df)

In [0]:
#write a manage delta table
df.write.format("delta").mode("overwrite").saveAsTable("training.default.delta_table")

In [0]:
%sql
select * from training.default.delta_table;

In [0]:
df = spark.read.format("delta").table("training.default.delta_table")
display(df)


In [0]:
# Append rows
new = spark.createDataFrame([(3,"carl",150)], cols)
new.write.format("delta").mode("append").saveAsTable("training.default.delta_table")

In [0]:
%sql
select * from training.default.delta_table;

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark,"training.default.delta_table")

updates = spark.createDataFrame([(2,"bob",250),(4,"dana",300)], cols)

In [0]:
delta_table.alias("t") \
    .merge(updates.alias("u"),"t.id=u.id") \
    .whenMatchedUpdate(set={"name":"u.name","amt":"u.amt"}) \
    .whenNotMatchedInsert(values={"id":"u.id","name":"u.name","amt":"u.amt"}) \
    .execute()

In [0]:
%sql
select * from training.default.delta_table version as of 0;

In [0]:
%sql
select * from training.default.delta_table

In [0]:
from pyspark.sql.functions import lit
# Example streaming source (rate) -> upsert to delta
stream = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

def upsert_microbatch(micro_df, epoch_id):
    # transform micro_df to proper schema (id, name, amt)
    updates = micro_df.selectExpr("value as id") \
                      .withColumn("name", lit("stream_user")) \
                      .withColumn("amt", lit(100))
    DeltaTable.forName(spark, "training.default.delta_table") \
      .alias("t").merge(updates.alias("s"), "t.id = s.id") \
      .whenMatchedUpdateAll() \
      .whenNotMatchedInsertAll().execute()

query = stream.writeStream.foreachBatch(upsert_microbatch).start()
query.awaitTermination(10)


In [0]:
emp_data =[(1,"pandit","Sales"),
           (2,"sneha","HR"),
           (3,"ravi","Finance")
           ]
columns= ["id","name","dept"]

emp_df =spark.createDataFrame(emp_data,columns)
display(emp_df)
emp_df.write.format("delta").mode("overwrite").saveAsTable("training.default.emp_df")

In [0]:
dept_data=[(1,"IT"),
           (2,"HR"),
           (3,"Finance")
           ]
dept_columns= ["id","dept"]

dept_df =spark.createDataFrame(dept_data,dept_columns)

In [0]:
%sql
select * from training.default.emp_df;

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark,"training.default.emp_df")

delta_table.alias("t") \
    .merge(dept_df.alias("s"),"t.id=s.id") \
    .whenMatchedUpdate(set={"dept":"s.dept"}) \
    .whenNotMatchedInsert(values={"id":"s.id","dept":"s.dept"}) \
    .execute()

In [0]:
%sql 
select * from training.default.emp_df;